<a href="https://colab.research.google.com/github/Kaitokidbua/ASEAN_Transport/blob/main/ASEAN_Part7_DevelopmentGap_Fig24-26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ── Setup ────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Dark Theme ────────────────────────────────────────────────────────────────
PAPER_BG   = '#0D1117'
PLOT_BG    = '#161B22'
GRID_C     = '#30363D'
FONT_C     = '#E6EDF3'
MUTED_C    = '#8B949E'
TEMPLATE   = 'plotly_dark'

CITY_COLORS = {
    'Bangkok':      '#FF6B6B',
    'Singapore':    '#74B9FF',
    'Kuala Lumpur': '#4ECDC4',
    'Jakarta':      '#FFB347',
    'Manila':       '#C77DFF',
}

def base_layout(height=460, legend_h=True):
    leg = dict(bgcolor='rgba(22,27,34,0.9)', bordercolor=GRID_C, borderwidth=1, font_size=11)
    if legend_h:
        leg.update(orientation='h', y=1.13, x=1, xanchor='right')
    return dict(
        template=TEMPLATE, paper_bgcolor=PAPER_BG, plot_bgcolor=PLOT_BG,
        font=dict(color=FONT_C, family='Segoe UI, Arial, sans-serif'),
        height=height, legend=leg,
        margin=dict(l=60, r=40, t=75, b=55),
        hoverlabel=dict(bgcolor='#1F2937', bordercolor=GRID_C, font_size=12),
    )

def ax_style():
    return dict(gridcolor=GRID_C, zerolinecolor=GRID_C)

print('✅ Setup complete — Dark theme loaded')

✅ Setup complete — Dark theme loaded


In [6]:
# ── Load Data ─────────────────────────────────────────────────────────────────
df = pd.read_csv('ASEAN_Urban_Growth_Final_with_Mode.csv')
df['Date']  = pd.to_datetime(df['Date'])
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

# ── Mode label maps ───────────────────────────────────────────────────────────
MODE_BKK = {'BTS':'BTS Skytrain','MRT':'MRT Blue/Purple','SRT':'SRT Red Line',
             'ARL':'Airport Rail Link','YL':'MRT Yellow Line','PK':'MRT Pink Line','RL':'Regional Rail'}
MODE_SGP = {'MRT':'MRT','Public Bus':'Public Bus','LRT':'LRT'}
MODE_KL  = {
    'rail_lrt_kj':'LRT Kelana Jaya','rail_mrt_kajang':'MRT Kajang',
    'rail_lrt_ampang':'LRT Ampang','bus_rkl':'RapidKL Bus',
    'rail_mrt_pjy':'MRT Putrajaya','rail_monorail':'KL Monorail',
    'rail_komuter':'KTM Komuter','rail_komuter_utara':'KTM Utara',
    'rail_ets':'ETS Train','rail_intercity':'Intercity Rail',
    'rail_tebrau':'KTM Tebrau','bus_rkn':'RapidKN Bus','bus_rpn':'RapidPN Bus'
}
MODE_JKT = {'TRANSJAKARTA':'TransJakarta (BRT)','KRL':'KRL Commuter',
             'MRT':'MRT Jakarta','LRT':'LRT Jakarta',
             'BUS SEKOLAH':'School Bus','KCI COMMUTER BANDARA':'Airport Rail','KAPAL':'Ferry'}

print(f'Dataset: {df.shape[0]:,} rows | {df["City"].nunique()} cities')
print('Cities:', df['City'].unique().tolist())

Dataset: 2,027 rows | 5 cities
Cities: ['Bangkok', 'Jakarta', 'Kuala Lumpur', 'Singapore', 'Manila']


In [8]:
# ── Chart 7.1: CAGR Bar ───────────────────────────────────────────────────────
yearly = df.groupby(['Year', 'City'])['Ridership'].sum().reset_index()
yearly = yearly.rename(columns={'Ridership': 'Total_Ridership'})

def calc_cagr(s, y0, y1):
    p = s.set_index('Year')['Total_Ridership']
    if y0 in p.index and y1 in p.index and p[y0]>0 and p[y1]>0:
        return ((p[y1]/p[y0])**(1/(y1-y0))-1)*100
    return np.nan

cagr_data = []
for city in yearly['City'].unique():
    v = calc_cagr(yearly[yearly['City']==city], 2021, 2024)
    if not np.isnan(v): cagr_data.append({'City':city,'CAGR':v})
cagr_df = pd.DataFrame(cagr_data).sort_values('CAGR')

sg_cagr = cagr_df[cagr_df['City']=='Singapore']['CAGR'].values[0]

fig = px.bar(cagr_df, x='CAGR', y='City', orientation='h',
    title='<b>ตารางที่ 7.1</b>  CAGR ผู้โดยสาร 2021–2024 (%)  |  ★ สิงคโปร์ = Developed Benchmark',
    labels={'CAGR':'CAGR (%)','City':''},
    color='City', color_discrete_map=CITY_COLORS, text_auto='.1f')
fig.add_vline(x=sg_cagr, line_dash='dot', line_color=CITY_COLORS['Singapore'],
              annotation_text='Singapore benchmark',
              annotation_position='top', annotation_font_color=CITY_COLORS['Singapore'],
              annotation_font_size=10)
fig.update_layout(**base_layout(380, legend_h=False), showlegend=False,
                  xaxis=dict(ticksuffix='%',**ax_style()), yaxis=ax_style())
fig.update_traces(textfont_size=11, textposition='outside')
fig.show()

# ── Chart 7.2: Passenger per Capita ──────────────────────────────────────────
ppc = df.groupby(['Year','City'])['Passenger_per_Capita'].sum().reset_index()
ppc = ppc[ppc['Year'].between(2019,2025)]
ppc = ppc[~((ppc['Year']==2025)&(ppc['City'].isin(['Kuala Lumpur','Manila'])))]

fig = px.line(ppc, x='Year', y='Passenger_per_Capita', color='City',
    markers=True, line_shape='spline',
    title='<b>ตารางที่ 7.2</b>  เที่ยวโดยสารต่อคนต่อปี (Passenger per Capita) — วัดระดับการพึ่งพา',
    labels={'Passenger_per_Capita':'เที่ยวต่อคนต่อปี','Year':''},
    color_discrete_map=CITY_COLORS)
fig.update_layout(**base_layout(480), hovermode='x unified',
                  xaxis=dict(dtick=1,**ax_style()), yaxis=ax_style())
fig.update_traces(line_width=2.5, marker_size=7)
fig.show()

# ── Chart 7.3: Growth Index (2021 = 100) ─────────────────────────────────────
base_yr = yearly[yearly['Year'].between(2021,2024)].copy()
idx_list=[]
for city in base_yr['City'].unique():
    sub = base_yr[base_yr['City']==city].sort_values('Year').copy()
    if 2021 in sub['Year'].values:
        base_val = sub[sub['Year']==2021]['Total_Ridership'].values[0]
        sub['Index'] = sub['Total_Ridership']/base_val*100
        idx_list.append(sub)
idx_df = pd.concat(idx_list)

fig = px.line(idx_df, x='Year', y='Index', color='City',
    markers=True, line_shape='spline',
    title='<b>ตารางที่ 7.3</b>  Growth Index (ปี 2021 = 100) — เปรียบเทียบอัตราเติบโตอย่างเป็นธรรม',
    labels={'Index':'Growth Index (2021=100)','Year':''},
    color_discrete_map=CITY_COLORS)
fig.add_hline(y=100, line_dash='dash', line_color=MUTED_C, line_width=1.2)
fig.update_layout(**base_layout(480), hovermode='x unified',
                  xaxis=dict(dtick=1,**ax_style()),
                  yaxis=dict(ticksuffix=' idx',**ax_style()))
fig.update_traces(line_width=2.5, marker_size=7)